# 02. Statistical Review

Regression uses the language of statistics. This chapter reviews the pieces needed for simple linear regression: data, samples, descriptive statistics, probability distributions, confidence intervals, and hypothesis tests.

## Data, populations, samples, and inference

A **population** is the full set of units or outcomes we care about. A **sample** is the portion we observe. A **parameter** is a numerical feature of the population, such as a population mean $\mu$ or true regression slope $\beta_1$. A **statistic** is a numerical feature computed from the sample, such as a sample mean $\bar y$ or fitted slope $b_1$.

**Statistical inference** uses sample information to learn about unknown population parameters. Regression is an inference tool because we usually do not know the true relationship. We estimate it from data.

## Descriptive statistics

For observations $y_1, y_2, \ldots, y_n$:

$$
\bar y = \frac{1}{n}\sum_{i=1}^{n}y_i
$$

is the sample mean, and

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(y_i-\bar y)^2
$$

is the sample variance. The sample standard deviation is $s = \sqrt{s^2}$. Other useful summaries include the median, mode, and range. Common displays include bar charts, pie charts, frequency tables, stem-and-leaf plots, and histograms.

Use the next cell to compute the summary values for a small grade data set.

## Browser package setup

Run the next cell before the first analysis cell. It loads the scientific Python packages used in this module into the browser-based Python kernel.

The first run in a browser session can take a little time. Later notebooks usually load faster because the browser can reuse cached packages.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from checks import check_close

grades = np.array([
    23, 62, 91, 83, 82, 64, 73, 94, 94, 52,
    67, 11, 87, 99, 37, 62, 40, 33, 80, 83,
    99, 90, 18, 73, 68, 75, 75, 90, 36, 55
])

grade_mean = grades.mean()
grade_variance = grades.var(ddof=1)
grade_sd = grades.std(ddof=1)

print(f"n = {len(grades)}")
print(f"sample mean = {grade_mean:.2f}")
print(f"sample variance = {grade_variance:.2f}")
print(f"sample standard deviation = {grade_sd:.2f}")

In [ ]:
print(check_close("sample mean", grade_mean, expected=66.53, tolerance=0.05))
print(check_close("sample variance", grade_variance, expected=630.60, tolerance=0.1))

## Normal distribution and standardization

Many regression formulas use the normal distribution as a starting point. If

$$
Y \sim N(\mu,\sigma^2),
$$

then the standardized value

$$
Z = \frac{Y-\mu}{\sigma}
$$

has a standard normal distribution with mean 0 and standard deviation 1.

In [ ]:
mu = 50
sigma = 15
a = 40
b = 70

z_a = (a - mu) / sigma
z_b = (b - mu) / sigma
prob_between = stats.norm.cdf(z_b) - stats.norm.cdf(z_a)

print(f"z_a = {z_a:.3f}")
print(f"z_b = {z_b:.3f}")
print(f"P({a} <= Y <= {b}) = {prob_between:.3f}")

## Central Limit Theorem

The Central Limit Theorem explains why averages are often easier to model than individual observations. If $Y_1,\ldots,Y_n$ are independent and identically distributed with mean $\mu$ and variance $\sigma^2$, then for large $n$:

$$
\bar Y \approx N\left(\mu, \frac{\sigma^2}{n}\right).
$$

This idea appears again in regression when we study the sampling distributions of fitted coefficients.

## Confidence intervals

A confidence interval gives a range of plausible values for an unknown parameter. The general form is:

$$
\text{estimate} \pm \text{critical value} \times \text{standard error}.
$$

For a population mean with unknown $\sigma$, we often use a $t$ critical value:

$$
\bar y \pm t_{\alpha/2,n-1}\frac{s}{\sqrt{n}}.
$$

In [ ]:
n = len(grades)
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
margin = t_crit * grade_sd / np.sqrt(n)
ci_mean = (grade_mean - margin, grade_mean + margin)

print(f"95% CI for the mean grade: ({ci_mean[0]:.2f}, {ci_mean[1]:.2f})")

## Hypothesis tests

A hypothesis test compares a null hypothesis $H_0$ with an alternative hypothesis $H_a$.

The basic workflow is:

1. state $H_0$ and $H_a$,
2. compute a test statistic,
3. compute or read a p-value,
4. compare the p-value to $\alpha$,
5. make a conclusion in context.

In regression, the most common first test is:

$$
H_0: \beta_1 = 0
$$

against

$$
H_a: \beta_1 \neq 0.
$$

If the slope is zero, the predictor does not explain a linear change in the mean response. If the p-value is small, we have evidence of a linear relationship.

## Reference distributions used in this module

Several later formulas use named sampling distributions:

- The **standard normal** distribution is used for a mean test when the population standard deviation is known.
- The **t distribution** is used when a standard deviation is estimated from the data, such as a one-sample mean with unknown $\sigma$, a two-sample pooled-variance mean test, or a regression coefficient test.
- The **chi-square distribution** is used for inference about one normal population variance.
- The **F distribution** is used for comparing two normal population variances and for regression ANOVA tests.

## Type I error, Type II error, and power

A hypothesis test can make two kinds of errors:

- **Type I error**: reject $H_0$ when $H_0$ is true. Its probability is controlled by $\alpha$.
- **Type II error**: fail to reject $H_0$ when a specific alternative value is true. Its probability is denoted by $\beta$.
- **Power** at that alternative value is $1-\beta$.

To compute a Type II error probability, first translate the rejection rule into an **acceptance region** for the statistic. Then evaluate the probability that the statistic falls in that acceptance region under the alternative parameter value.

For example, in a two-sided known-$\sigma$ test for a mean,

$$
Z = \frac{\bar X - \mu_0}{\sigma/\sqrt{n}}
$$

and the non-rejection region is

$$
-z_{\alpha/2} \le Z \le z_{\alpha/2}.
$$

Equivalently, in terms of $\bar X$,

$$
\mu_0 - z_{\alpha/2}\frac{\sigma}{\sqrt n}
\le \bar X \le
\mu_0 + z_{\alpha/2}\frac{\sigma}{\sqrt n}.
$$

If the true mean is some alternative value $\mu_a$, compute the probability that $\bar X$ falls in that interval using

$$
\bar X \sim N\left(\mu_a, \frac{\sigma^2}{n}\right).
$$

## Two-sample means with equal variances

For two independent normal samples with equal population variances, the pooled variance is

$$
s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}.
$$

To test

$$
H_0: \mu_1 - \mu_2 = \Delta_0,
$$

use

$$
t = \frac{(\bar x_1-\bar x_2)-\Delta_0}{s_p\sqrt{1/n_1+1/n_2}},
$$

with $n_1+n_2-2$ degrees of freedom. For a two-sided alternative, the p-value is

$$
2P\left(T_{n_1+n_2-2} \ge |t|\right).
$$

## Comparing two population variances

For independent normal samples, a variance ratio follows an F distribution under the null hypothesis:

$$
F = \frac{s_1^2}{s_2^2} \sim F_{n_1-1,n_2-1}
\quad \text{when } H_0: \sigma_1^2=\sigma_2^2 \text{ is true.}
$$

For a two-sided test, use both tails of the F distribution. A common computational approach is to place the larger sample variance in the numerator and double the upper-tail probability, while keeping the degrees of freedom matched to the numerator and denominator.


In [ ]:
# Reference distribution calculator: change the numbers to match a problem you are solving.
alpha = 0.05
example_t_df = 10
example_f_df_num = 5
example_f_df_den = 12

print("Two-sided t critical value:", round(stats.t.ppf(1 - alpha / 2, df=example_t_df), 4))
print("Two-sided F lower critical value:", round(stats.f.ppf(alpha / 2, dfn=example_f_df_num, dfd=example_f_df_den), 4))
print("Two-sided F upper critical value:", round(stats.f.ppf(1 - alpha / 2, dfn=example_f_df_num, dfd=example_f_df_den), 4))


## What to remember

Regression software will compute many quantities for you. You still need to know what they mean:

- estimates are sample-based guesses for unknown parameters,
- standard errors measure sampling uncertainty,
- confidence intervals give plausible parameter values,
- p-values measure evidence against a null hypothesis,
- statistical significance is not the same as practical importance.